# الفصل 42: معايير القياس المرجعية: Benchmarks

> عندما تقرأ "النموذج X حقّق 85% على MMLU"، يجب أن تفهم بالضبط ماذا يعني هذا، وما حدوده، ولماذا رقم واحد لا يكفي أبدًا للحكم على نموذج شامل.

- **الجزء:** 9 — التقييم الشامل
- **الفصل:** 42
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 41: أسس القياس الكمّي
- **الوسوم:** MMLU، GSM8K، HellaSwag، Contamination

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch42-benchmarks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch42-benchmarks.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch42-benchmarks.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> عندما تقرأ "النموذج X حقّق 85% على MMLU"، يجب أن تفهم بالضبط ماذا يعني هذا، وما حدوده، ولماذا رقم واحد لا يكفي أبدًا للحكم على نموذج شامل.

> **بيانات الفصل**
>
> - **الجزء:** 9 — التقييم الشامل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 41: أسس القياس الكمّي


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم MMLU — المعيار الأكثر استشهادًا (ولماذا يجب الحذر منه)
- فهم مشكلة تلوث البيانات (Benchmark Contamination) بالتفصيل العملي
- فهم MMLU


## 1. لماذا يهمّك هذا الموضوع؟

عندما تقرأ "النموذج X حقّق 85% على MMLU"، يجب أن تفهم **بالضبط** ماذا يعني هذا، وما حدوده، ولماذا رقم واحد لا يكفي أبدًا للحكم على نموذج شامل.


## 2. الخريطة الذهنية


```text
Benchmarks الشاملة
    │
    ├── المعرفة العامة والاستدلال
    │   ├── MMLU (57 مادة أكاديمية)
    │   ├── MMLU-Pro (نسخة أصعب، مشتتات أكثر)
    │   └── BIG-Bench Hard (BBH)
    │
    ├── الاستدلال الرياضي والمنطقي
    │   ├── GSM8K (رياضيات ابتدائية-متوسطة)
    │   ├── MATH (رياضيات متقدمة)
    │   └── ARC (استدلال علمي)
    │
    ├── البرمجة
    │   ├── HumanEval
    │   ├── MBPP
    │   └── SWE-bench (مهام هندسة برمجيات حقيقية)
    │
    ├── المحادثة والتفضيل البشري
    │   ├── Chatbot Arena (Elo تنافسي)
    │   ├── MT-Bench
    │   └── AlpacaEval
    │
    └── الأمان والمحاذاة
        ├── TruthfulQA
        └── ToxiGen
```


## 3. الشرح المتدرج

### 🟢 أ) MMLU — المعيار الأكثر استشهادًا (ولماذا يجب الحذر منه)

MMLU (Massive Multitask Language Understanding) يختبر 57 مادة (من القانون للطب للفيزياء) بأسئلة اختيار من متعدد.


In [ ]:
def evaluate_mmlu_style(model, tokenizer, questions: list, choices_list: list, answers: list):
    """
    التقييم القياسي لأسئلة الاختيار من متعدد:
    نقارن log-likelihood لكل خيار، ونختار الأعلى (وليس التوليد الحر!)
    """
    correct = 0
    for question, choices, correct_answer_idx in zip(questions, choices_list, answers):
        choice_scores = []
        for choice in choices:
            prompt = f"{question}\nالإجابة: {choice}"
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids
            with torch.no_grad():
                outputs = model(input_ids, labels=input_ids)
            # نستخدم negative loss كـ score (أعلى = احتمال أعلى)
            choice_scores.append(-outputs.loss.item())

        predicted_idx = choice_scores.index(max(choice_scores))
        if predicted_idx == correct_answer_idx:
            correct += 1

    return correct / len(questions)


> ⚠️ **مصيدة منهجية مهمة**: الطريقة أعلاه (مقارنة log-likelihood) تُعطي نتائج مختلفة جوهريًا عن توليد حر ثم مطابقة الحرف (A/B/C/D). **كن حذرًا جدًا دومًا من مقارنة أرقام MMLU بين أوراق بحثية مختلفة** إذا لم تستخدم نفس البروتوكول بالضبط (0-shot مقابل 5-shot، log-likelihood مقابل generation matching) — هذا سبب شائع جدًا لتضارب الأرقام المُعلنة لنفس النموذج بين مصادر مختلفة.

### 🟡 ب) مشكلة تلوث البيانات (Benchmark Contamination) بالتفصيل العملي

راجعنا هذا بإيجاز في الفصل 18 أثناء تنظيف البيانات. الآن نتعمّق في **كشفه بعد التدريب**:


In [ ]:
def detect_contamination_via_perplexity_outlier(model, tokenizer, benchmark_samples: list,
                                                   reference_corpus_samples: list):
    """
    فكرة: إذا كان نموذج قد "حفظ" عيّنات benchmark أثناء التدريب،
    فإن perplexity عليها ستكون أقل بشكل غير طبيعي مقارنة بنصوص مشابهة لم يرها
    """
    def avg_ppl(samples):
        ppls = []
        for text in samples:
            input_ids = tokenizer(text, return_tensors="pt").input_ids
            with torch.no_grad():
                loss = model(input_ids, labels=input_ids).loss
            ppls.append(torch.exp(loss).item())
        return sum(ppls) / len(ppls)

    benchmark_ppl = avg_ppl(benchmark_samples)
    reference_ppl = avg_ppl(reference_corpus_samples)

    contamination_ratio = reference_ppl / benchmark_ppl
    # إذا كانت > 1.5-2x، هذا مؤشر قوي على احتمال التلوث (حفظ، وليس تعميم حقيقي)
    return {
        "benchmark_ppl": benchmark_ppl,
        "reference_ppl": reference_ppl,
        "contamination_suspicion_ratio": contamination_ratio
    }

def ngram_overlap_contamination_check(training_corpus_sample: list, benchmark_texts: list, n=13):
    """
    الطريقة الأكثر شيوعًا في التقارير الرسمية (مثل GPT-4 technical report):
    فحص تطابق n-grams طويلة (13 كلمة فأكثر) بين بيانات التدريب والـ benchmark
    """
    def get_ngrams(text, n):
        tokens = text.split()
        return set(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

    contaminated_count = 0
    for benchmark_text in benchmark_texts:
        benchmark_ngrams = get_ngrams(benchmark_text, n)
        for train_text in training_corpus_sample:
            train_ngrams = get_ngrams(train_text, n)
            if benchmark_ngrams & train_ngrams:   # أي تقاطع يُعتبر تلوثًا محتملًا
                contaminated_count += 1
                break

    return contaminated_count / len(benchmark_texts)


> 💡 **نصيحة عملية**: الممارسة الأفضل في الصناعة اليوم هي **إصدار نسخ جديدة** من Benchmarks بشكل دوري (مثل LiveCodeBench الذي يضيف مسائل برمجية جديدة باستمرار من مسابقات حديثة) لتفادي مشكلة التلوث التراكمي مع مرور الوقت وتسرّب البيانات عبر الإنترنت.

### 🔴 ج) طبقة التعمّق: Benchmarks الاستدلال المتقدم ونماذج o1/DeepSeek-R1

مع ظهور **نماذج الاستدلال** (o1, o3, DeepSeek-R1, QwQ) التي تُنفق "وقت تفكير" إضافي أثناء الاستدلال (Test-Time Compute)، ظهرت حاجة لـ Benchmarks تتحدى **الاستدلال العميق متعدد الخطوات** وليس فقط الاسترجاع المعرفي:

| Benchmark | ما يختبره | لماذا يتحدّى نماذج الاستدلال |
|---|---|---|
| **GPQA (Diamond)** | أسئلة فيزياء/كيمياء/أحياء بمستوى دكتوراه | حتى خبراء غير متخصصين يفشلون؛ يتطلب استدلالًا علميًا عميقًا |
| **ARC-AGI** | ألغاز بصرية-منطقية مجردة | يختبر التعميم الحقيقي وليس الحفظ النمطي |
| **FrontierMath** | مسائل رياضية بحثية صعبة جدًا | حتى أفضل النماذج تحقق نسب نجاح منخفضة جدًا (<5%) |
| **AIME** | مسائل مسابقة رياضية أمريكية | قياس معياري قوي لتحسّن الاستدلال خطوة بخطوة |


In [ ]:
def evaluate_with_chain_of_thought(model, tokenizer, question: str, use_reasoning_tokens=True):
    """
    تقييم أسلوب نماذج الاستدلال الحديثة: السماح بـ "مساحة تفكير" قبل الإجابة النهائية
    """
    if use_reasoning_tokens:
        prompt = f"""السؤال: {question}
فكّر خطوة بخطوة بعناية شديدة قبل الإجابة النهائية.
<تفكير>"""
    else:
        prompt = f"السؤال: {question}\nالإجابة المباشرة:"

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    output_ids = model.generate(
        input_ids, max_new_tokens=2048,   # مساحة أكبر بكثير لنماذج الاستدلال
        temperature=0.0, do_sample=False
    )
    full_response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # استخراج الإجابة النهائية بعد علامة الانتقال (حسب تنسيق النموذج)
    if "<إجابة_نهائية>" in full_response:
        final_answer = full_response.split("<إجابة_نهائية>")[-1].strip()
    else:
        final_answer = full_response.strip()
    return full_response, final_answer


> 💡 **ملاحظة محورية**: تقييم نماذج الاستدلال يتطلب **ميزانية حسابية متغيرة** (variable compute budget) — بعض الأسئلة تحتاج آلاف الـ tokens من "التفكير الداخلي" قبل الوصول لإجابة صحيحة. هذا يُدخل بُعدًا جديدًا في التقييم: **منحنى الأداء مقابل ميزانية الاستدلال** (Performance vs. Test-Time Compute curve)، وليس فقط دقة واحدة ثابتة.


## 4. مثال محلول خطوة بخطوة

لنقارن بروتوكولين على نفس السؤال:

**السؤال:** ما عاصمة أستراليا؟ (A) سيدني (B) ملبورن (C) كانبرا (D) بيرث.

**البروتوكول 1 — log-likelihood:** نمرّر `السؤال + الخيار` ونقارن loss لكل خيار. النموذج قد يفضّل `سيدني` لأنها الأشهر في النصوص، حتى لو كان يعرف أن الإجابة كانبرا.

**البروتوكول 2 — توليد حر:** نطلب `أجب بالحرف فقط` ونقرأ أول رمز. النموذج قد يُخرج `C` صحيحة، لكنه قد يُخرج شرحًا طويلًا يبدأ بـ`العاصمة هي...` فيفشل الاستخراج.

**النتيجة:** نفس النموذج قد يحصل على 78% في الأول و84% في الثاني. هذا ليس تحسّنًا حقيقيًا؛ إنه اختلاف بروتوكول.

**القاعدة:** عند نشر نتيجة MMLU، وثّق: عدد الأمثلة (0-shot/5-shot)، طريقة حساب الاحتمال، prompt template، وطريقة استخراج الحرف. بدون هذه التفاصيل، الرقم غير قابل للتكرار.


## 5. نصائح وحيل

> 💡 **ثبّت prompt template**: تغيير صياغة السؤال يغيّر النتيجة بعدة نقاط. استخدم قالبًا واحدًا لكل التجارب.

> 💡 **استخدم 5-shot عند المقارنة بنتائج منشورة**: معظم أرقام MMLU الرسمية 5-shot. المقارنة بـ0-shot غير عادلة.

> 💡 **أبلغ عن فترات ثقة**: 1000 سؤال يعطي هامش خطأ ~±1.5%. لا تفرح بفرق 0.5%.

> 💡 **افحص التلوث**: قارن perplexity على benchmark مقابل corpus مرجعي، وابحث عن 13-gram overlap.

> 💡 **أضف benchmark عربيًا**: MMLU إنجليزي؛ أضف ArabicMMLU أو مجموعة داخلية لقياس اللغة.

> 💡 **اختبر نماذج الاستدلال بميزانية متغيرة**: لا تختبر o1/R1 بـmax_tokens صغير؛ أعطها مساحة تفكير.


## 6. أخطاء شائعة ومصائد

> ⚠️ **مقارنة أرقام من بروتوكولات مختلفة**: 0-shot log-likelihood مقابل 5-shot generation ليست نفس المقياس. وثّق دائمًا.

> ⚠️ **تلوث benchmark**: إذا كان جزء من بيانات الاختبار في التدريب، النتيجة منتفخة. افحص n-gram overlap ولا تعتمد على رقم واحد.

> ⚠️ **الاعتماد على MMLU وحده**: MMLU يقيس المعرفة الأكاديمية، لا البرمجة أو السلامة أو العربية. استخدم مجموعة متنوعة.

> ⚠️ **إهمال صعوبة الأسئلة**: متوسط 85% قد يخفي فشلًا كاملًا في فئة معينة. حلّل حسب المادة/اللغة.

> ⚠️ **اختيار النموذج حسب benchmark واحد**: هذا يخلق Goodhart's Law. اربط التقييم بحالة الاستخدام.


## 7. جدول مقارنة

| الهدف من التقييم | Benchmark المناسب |
|---|---|
| معرفة عامة واسعة | MMLU-Pro |
| استدلال رياضي أساسي | GSM8K |
| استدلال رياضي متقدم جدًا | AIME, FrontierMath |
| قدرات برمجية عملية | SWE-bench, HumanEval |
| جودة محادثة عامة من منظور بشري | Chatbot Arena, MT-Bench |
| الصدق وتجنّب المعلومات المضلّلة | TruthfulQA |
| أمان ومحتوى ضار | ToxiGen, SafetyBench |


## 8. ملخص أفضل الممارسات

- [ ] استخدم مجموعة متنوعة من Benchmarks تغطي أبعادًا مختلفة (معرفة، استدلال، أمان، محادثة)
- [ ] وثّق بدقة إعدادات التقييم (n-shot، طريقة الاستخراج، درجة الحرارة)
- [ ] افحص التلوث المحتمل قبل نشر أي نتيجة رسمية
- [ ] لا تعتمد على Benchmark واحد مهما كان مشهورًا (مصيدة "Goodhart's Law": عندما يصبح المقياس هدفًا، يتوقف عن كونه مقياسًا جيدًا)
- [ ] أضف دائمًا Benchmark خاص بمجالك/لغتك (Domain-specific/Arabic-specific)

---


## 9. أسئلة مقابلات

**س1: ما هي Goodhart's Law وكيف تنطبق على تقييم LLMs؟**

**الإجابة النموذجية:** "عندما يصبح المقياس هدفًا، يتوقف عن كونه مقياسًا جيدًا". إذا حسّنت الفرق نماذجها خصيصًا لـMMLU (حتى عبر بيانات مسربة)، تفقد الدرجة دلالتها كدليل على الذكاء العام. الحل: تنويع المقاييس، تحديثها، وقياس out-of-distribution.

**س2: كيف تكشف تلوث benchmark بعد التدريب؟**

**الإجابة النموذجية:** عبر فحص n-gram overlap (13-gram) بين بيانات التدريب والاختبار، ومقارنة perplexity على benchmark مقابل corpus مرجعي مشابه. إذا كانت perplexity منخفضة بشكل غير طبيعي، فهناك اشتباه. الأفضل: benchmarks متجددة مثل LiveCodeBench.

**س3: لماذا لا يكفي MMLU للحكم على نموذج عربي؟**

**الإجابة النموذجية:** MMLU إنجليزي في معظمه ويقيس معرفة أكاديمية. نموذج قد يتفوق فيه لكنه ضعيف في اللهجات أو النحو العربي أو المهام المحلية. تحتاج ArabicMMLU واختبارات لهجية ومهام مجال عربي.


## 10. تمرين عملي

1. قيّم نموذجًا صغيرًا (Qwen2.5-0.5B) على 20 سؤال GSM8K بطريقتين: إجابة مباشرة وCoT. قارن الدقة.
2. كرّر التقييم بـ3 صيغ prompt مختلفة، ولاحظ التباين.
3. افحص 13-gram overlap بين 100 نص تدريبي و20 سؤال benchmark.
4. حلّل نتائج MMLU حسب المادة (إن توفرت) وابحث عن فئات ضعيفة.
5. اكتب بروتوكول تقييم من صفحة واحدة يمكن لفريقك تكراره.


## 11. مراجع للتعمّق

- Hendrycks, D., et al. (2021). *Measuring Massive Multitask Language Understanding* (MMLU).
- Cobbe, K., et al. (2021). *Training Verifiers to Solve Math Word Problems* (GSM8K).
- Rein, D., et al. (2023). *GPQA: A Graduate-Level Google-Proof Q&A Benchmark*.
- Chollet, F. (2019). *On the Measure of Intelligence* (ARC).
- Srivastava, A., et al. (2023). *Beyond the Imitation Game: Quantifying and Extrapolating the Capabilities of Language Models* (BIG-Bench).


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التقييم الشامل / الفصل 42: معايير القياس المرجعية: Benchmarks.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
